# Pré Processamento do Conjunto de Dados UEyes

## Bibliotecas

In [5]:
# bibliotecas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import re
from PIL import Image

1º passo: juntar os dados em um único arquivo parquet.

In [6]:
# caminhos dos arquivos

# registros de rastreamento ocular
log_files = "../data/UEyes_dataset/UEyes_dataset/eyetracker_logs"

# arquivos de imagens
img_files = '../data/UEyes_dataset/UEyes_dataset/images'

# categorias de imagens
cat_img_file = '../data/UEyes_dataset/UEyes_dataset/image_types.csv'

As colunas de interesse do conjunto de dados são:
- `MEDIA_NAME`: O nome do arquivo de imagem estática da UI.
- `TIME`: tempo, em segundos, da ocorrência do evento de fixação do olhar.
- `FPOGD`: Duração da fixação, em segundos.
- `FPOGX`, `FPOGY`: Posição do olhar fixo do usuário nas coordenadas x e y, respectivamente, do monitor na escala normalizada [0,1].
- `FPOGV`: Flag de validade do registro de fixação.
- `SACCADE_MAG`: Amplitude da sacada (distância entre dois pontos de fixação consecutivos), em pixels.
- `LPMM`, `RPMM`: Tamanho, em milímetros das pupilas esquerda e direita, respectivamente.
- `LPMMV`, `RPMMV`: Flag de validade do registro do tamanho das pupilas esquerda e direita, respectivamente.
- `participant_id`: Número único de cada participante do experimento.
- `block_id`: Número do bloco visualizado pelo participante.

In [12]:
# lista de todos os arquivos de log
all_files = [f for f in os.listdir(log_files) if f.endswith('_fixations.csv')]

print(f" -> Listando arquivos do diretório de logs.\nTOTAL: {len(all_files)} arquivos.\n")

# colunas para leitura
cols_to_load = [
    "MEDIA_NAME", "FPOGD", "FPOGX", "FPOGY", "FPOGV",
    "SACCADE_MAG", "LPMM", "RPMM", "LPMMV", "RPMMV"
]

# lista de dataframes provenientes dos arquivos csv
df_list =[]

# abertura e cômputo dos arquivos de log
for filename in all_files:
    match = re.match(r'(\d+)_([Kk][Hh]\d+)_fixations.csv', filename)
    if not match: continue

    # definição do bloco e do id do participante
    block_id, participant_id = match.groups()
    # caminho completo do arquivo
    file_path = os.path.join(log_files, filename)

    try:
        # leitura do csv bruto
        df = pd.read_csv(
            file_path, 
            usecols=lambda c: c in cols_to_load or c.startswith("TIME"))
        
        # encontra colunas que iniciam com TIME
        time_cols = [c for c in df.columns if c.startswith("TIME")]

        # verifica se foram puxadas colunas TIME
        if time_cols:
            time_col_original = time_cols[0]

            # renomeia a coluna original
            df.rename(columns={time_col_original:"TIME"}, inplace=True)

            # drop em qualquer coluna time restante
            df.drop(columns=time_cols[1:], inplace=True)
        
        else:
            print(f"AVISO: Nenhuma coluna de tempo encontrada no arquivo {filename}")

        df_temp = df.copy()

        # adição de ids de participante e bloco
        df_temp["participant_id"] = participant_id
        df_temp["block_id"] = block_id

        # lista de dataframes para depois concatenar
        df_list.append(df_temp)

    except Exception as e:
        print(f"Erro no arquivo {filename}: {e}")

# dataset total
df_raw_full = pd.concat(df_list, ignore_index=True)

# número de arquivos lidos
n_files_read = len(df_raw_full.drop_duplicates(subset=['participant_id', 'block_id']))

print(f" -> Total de arquivos carregados: {n_files_read}\n")
print(f" -> Total de linhas carregadas (dataset bruto): {len(df_raw_full)}\n")
# verificando valores nan
nan_df_raw = df_raw_full.isna().sum().sum()
print(f" -> Total de valores NaN: {nan_df_raw}\n")

# reordenando colunas
#colunas da frente
cols_front = ['participant_id', 'block_id', 'MEDIA_NAME', 'TIME']

# colunas restantes
cols_remaining = [c for c in df_raw_full.columns if c not in cols_front]

# concatenação
new_order = cols_front + cols_remaining

# reatribuição
df_raw_full = df_raw_full[new_order]

# salvando em parquet
print(" -> Salvando em formato parquet...\n")

df_raw_full.to_parquet('../data/raw_dataset.parquet', engine='pyarrow', compression='snappy')

print(df_raw_full.info())

 -> Listando arquivos do diretório de logs.
TOTAL: 554 arquivos.

 -> Total de arquivos carregados: 554

 -> Total de linhas carregadas (dataset bruto): 351323

 -> Total de valores NaN: 0

 -> Salvando em formato parquet...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351323 entries, 0 to 351322
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   participant_id  351323 non-null  object 
 1   block_id        351323 non-null  object 
 2   MEDIA_NAME      351323 non-null  object 
 3   TIME            351323 non-null  float64
 4   FPOGX           351323 non-null  float64
 5   FPOGY           351323 non-null  float64
 6   FPOGD           351323 non-null  float64
 7   FPOGV           351323 non-null  int64  
 8   LPMM            351323 non-null  float64
 9   LPMMV           351323 non-null  int64  
 10  RPMM            351323 non-null  float64
 11  RPMMV           351323 non-null  int64  
 12  SACCADE_MAG   

2º passo: construir conjunto de dados de imagens com nome, categoria e resolução.

In [13]:
# carregando parquet
df_raw_full = pd.read_parquet('../data/raw_dataset.parquet', engine='pyarrow')
df_raw_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351323 entries, 0 to 351322
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   participant_id  351323 non-null  object 
 1   block_id        351323 non-null  object 
 2   MEDIA_NAME      351323 non-null  object 
 3   TIME            351323 non-null  float64
 4   FPOGX           351323 non-null  float64
 5   FPOGY           351323 non-null  float64
 6   FPOGD           351323 non-null  float64
 7   FPOGV           351323 non-null  int64  
 8   LPMM            351323 non-null  float64
 9   LPMMV           351323 non-null  int64  
 10  RPMM            351323 non-null  float64
 11  RPMMV           351323 non-null  int64  
 12  SACCADE_MAG     351323 non-null  float64
dtypes: float64(7), int64(3), object(3)
memory usage: 34.8+ MB


In [14]:
print(' -> Mapeamento das imagens...\n')

# csv com o nome das imagens e a categoria de cada uma
df_cat = pd.read_csv(cat_img_file, sep=';', engine='python')

print(f' -> Número de imagens: {df_cat['Image Name'].nunique()}')
print(df_cat['Category'].value_counts())
print()

# função para capturar dimensões das imagens
def get_dimensions(filename, img_dir):
    """
    Função para adquirir largura e altura de uma imagem.
    """
    try:
        path = os.path.join(img_dir, filename)
        with Image.open(path) as img:
            return img.size # tupla (largura, altura)
    except Exception:
        return (None, None)
    
# captura de resoluções
resolutions = [get_dimensions(name, img_files) for name in df_cat['Image Name']]

df_cat[['width', 'height']] = pd.DataFrame(resolutions, index=df_cat.index)

nan_in_width = df_cat['width'].isna().sum()
nan_in_height = df_cat['height'].isna().sum()
print(f" -> Valores nulos:\n\tWidth: {nan_in_width}\n\tHeight: {nan_in_height}\n")

print("Salvando em formato parquet...")
df_cat.to_parquet('../data/img_types_sizes.parquet', engine='pyarrow', compression='snappy')

 -> Mapeamento das imagens...

 -> Número de imagens: 1980
Category
desktop    495
mobile     495
poster     495
web        495
Name: count, dtype: int64

 -> Valores nulos:
	Width: 0
	Height: 0

Salvando em formato parquet...
